In [1]:
print("test")

test


In [2]:
from pathlib import Path
import sys

# Ensure the project src/ directory is importable from the notebook.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in [_cwd, *_cwd.parents] if (p / "src" / "timeseries_forecast").exists()
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from timeseries_forecast.data_prep.raw_data import read_raw_alu_data
from timeseries_forecast.data_prep.pipeline import run_data_pipeline
from timeseries_forecast.data_prep.constants import RawAluColumn
# from timeseries_forecast.feature import autoregressive_features
# from timeseries_forecast.eval.eval import evaluate, evaluate_predictions
# from timeseries_forecast.feature.custom_features import add_log_ret_features
# from timeseries_forecast.feature import engineer_features
# from timeseries_forecast.config.feature_config import FeatureConfig
# from timeseries_forecast.data_prep.data_resample import resample_dataframe_with_aggregations

In [3]:
alu_df = read_raw_alu_data()
alu_df.Date

C:\Users\Acer\Desktop\OTH\FPP\fpp\src\timeseries_forecast\data_prep\raw_data.py:35: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[RawAluColumn.OPEN_TIME] = pd.to_datetime(df[RawAluColumn.OPEN_TIME])


0      2024-08-29
1      2024-08-28
2      2024-08-27
3      2024-08-26
4      2024-08-23
          ...    
2730   2014-01-07
2731   2014-01-06
2732   2014-01-03
2733   2014-01-02
2734   2014-01-01
Name: Date, Length: 2735, dtype: datetime64[us]

In [4]:
alu_df_cleaned = run_data_pipeline(
    read_raw_alu_data,
    timestamp_column=RawAluColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawAluColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawAluColumn.VOLUME],
)
alu_df_cleaned.head()


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
⚠ 4 Zeilen sind doppelt (identischer Inhalt)
  Beispiele (erste 5 Timestamps):
DatetimeIndex(['2020-09-29', '2020-09-30', '2024-07-30', '2024-07-31'], dtype='datetime64[us]', name='RawAluColumn.OPEN_TIME', freq=None)

Fehlende Werte Check
⚠ Es gibt 1159 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 15 NaN-Werte insgesamt gefunden:
  Vol.: 15 (0.55%)

0-Werte Check
⚠ 32 0-Werte insgesamt gefunden:
  Vol.: 32 (1.17%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,894 Einträge
  Aktueller DataFrame:    2,735 Zeilen
  Zu ergänzende Einträge: 1,159
    RawAluColumn.VOLUME: 32 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1159 → 0 NaN
    Open: 1159 → 0 NaN
    High: 1159 → 0 NaN
    Low: 1159 → 0 NaN
    Vol.: 1206 → 0 NaN
    Change %: 1159 → 0 NaN

  Shape vorher:  (2735, 6)
  Shape nachher: (3894, 6)
✓ Interpolation abgeschlossen!

Duplikat-Che

C:\Users\Acer\Desktop\OTH\FPP\fpp\src\timeseries_forecast\data_prep\raw_data.py:35: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[RawAluColumn.OPEN_TIME] = pd.to_datetime(df[RawAluColumn.OPEN_TIME])


,Price,Open,High,Low,Vol.,Change %,Mode
2014-01-01,110.10,110.000000,110.70,109.900000,450.000000,0.640000,Train
2014-01-02,111.20,110.550000,112.20,110.450000,4720.000000,1.000000,Train
2014-01-03,109.20,110.900000,111.10,109.050000,4520.000000,-1.800000,Train
2014-01-04,109.35,110.433333,110.65,108.733333,4333.333333,-1.063333,Train
2014-01-05,109.50,109.966667,110.20,108.416667,4146.666667,-0.326667,Train


In [5]:
from pathlib import Path

cleaned_dir = PROJECT_ROOT / "data" / "cleaned"
cleaned_dir.mkdir(parents=True, exist_ok=True)

output_path = cleaned_dir / "metals_aluminium_cleaned.csv"
alu_df_cleaned.to_csv(output_path, index=True)

print(f"Saved cleaned dataframe to: {output_path}")

Saved cleaned dataframe to: C:\Users\Acer\Desktop\OTH\FPP\fpp\data\cleaned\metals_aluminium_cleaned.csv
